In [ ]:
# # one-time setup
# # installing cmake
# !wget https://github.com/Kitware/CMake/releases/download/v3.31.4/cmake-3.31.4-linux-x86_64.sh
# !chmod +x cmake-3.31.4-linux-x86_64.sh
# !sudo ./cmake-3.31.4-linux-x86_64.sh --prefix=/usr/local --skip-license 
# !rm cmake-3.31.4-linux-x86_64.sh
# !cmake --version


# # installing cuda
# # !sudo apt remove --purge '^nvidia-.*' -y
# !sudo apt autoremove -y
# !wget https://developer.download.nvidia.com/compute/cuda/repos/ubuntu$(lsb_release -rs | tr -d '.')/x86_64/cuda-keyring_1.0-1_all.deb
# !sudo dpkg -i cuda-keyring_1.0-1_all.deb
# # !rm cuda-keyring_1.0-1_all.deb
# !sudo apt update
# !sudo apt install cuda -y
# !echo 'export PATH=/usr/local/cuda/bin:$PATH' >> ~/.bashrc
# !echo 'export LD_LIBRARY_PATH=/usr/local/cuda/lib64:$LD_LIBRARY_PATH' >> ~/.bashrc
# !source ~/.bashrc


# # stable_diffusion_cpp_python
# !sudo apt install build-essential g++-11 ninja-build
# !sudo update-alternatives --install /usr/bin/gcc gcc /usr/bin/gcc-11 100
# !sudo update-alternatives --install /usr/bin/g++ g++ /usr/bin/g++-11 100
# !CMAKE_ARGS="-DCUDA_ARCH=sm_89 -DSD_CUDA=ON -DSD_FLASH_ATTN=ON" pip install stable-diffusion-cpp-python --upgrade --force-reinstall --no-cache-dir

In [ ]:
%pip install numpy torch
%pip install flash-attn --no-build-isolation
%pip install transformers accelerate optimum auto-gptq huggingface-hub huggingface_hub[hf_transfer] pickleshare safetensors \
                pymongo json5 diffusers protobuf peft hf_transfer --upgrade

%pip install xformers "xfuser[diffusers,flash-attn]"

In [ ]:
%%capture
# Setting up HF transfer
from huggingface_hub import login
import base64
t = 'aGZfaHZqck9VTXFvTXF3dW9HR3JoTlZKSWlsZUtFTlNQbXRjTw=='
k = base64.b64decode(t.encode()).decode()
login(token=k, add_to_git_credential=True)
%env HUGGINGFACEHUB_API_TOKEN={k}
%env HF_TOKEN={k}
%env HF_HUB_ENABLE_HF_TRANSFER=1

In [ ]:
# in terminal pip install hf_transfer
# !huggingface-cli download black-forest-labs/FLUX.1-dev ae.safetensors
# !huggingface-cli download comfyanonymous/flux_text_encoders clip_l.safetensors
# !huggingface-cli download comfyanonymous/flux_text_encoders t5xxl_fp16.safetensors
# !huggingface-cli download kudzueye/boreal-flux-dev-v2 boreal-v2.safetensors
# !huggingface-cli download kudzueye/Boreal boreal-flux-dev-lora-v04_1000_steps.safetensors
# !huggingface-cli download Shakker-Labs/FLUX.1-dev-LoRA-AntiBlur FLUX-dev-lora-AntiBlur.safetensors

!huggingface-cli download black-forest-labs/FLUX.1-dev

In [ ]:
import torch
from diffusers import FluxPipeline
from huggingface_hub import hf_hub_download
base_model_id = "black-forest-labs/FLUX.1-dev"
repo_name = "ByteDance/Hyper-SD"
ckpt_name = "Hyper-FLUX.1-dev-8steps-lora.safetensors"
pipe = FluxPipeline.from_pretrained(base_model_id, dtype=torch.float16, device_map="balanced", max_memory={0: "24GB", 1: "24GB", 2: "24GB", 3: "24GB"})
pipe.load_lora_weights(hf_hub_download(repo_name, ckpt_name))
pipe.fuse_lora(lora_scale=0.125)
# pipe.to("cpu", dtype=torch.float16)
seed = 1000
image=pipe(
    prompt="a photo of a cat",
    num_inference_steps=8,
    guidance_scale=3.5,
    generator=torch.Generator("cuda").manual_seed(seed),
).images[0]
image.save("output.png")

In [ ]:
globals()["loaded_models"]

In [ ]:
import torch, time
from diffusers import FluxPipeline, FluxTransformer2DModel, AutoencoderKL
from diffusers.image_processor import VaeImageProcessor
from huggingface_hub import hf_hub_download

def generate(guidance_scale=0):
    global loaded_models
    
    if "loaded_models" not in globals():
        loaded_models = {}

    if "encoder" not in loaded_models:
        print("Loading encoder pipeline...")
        loaded_models["encoder"] = FluxPipeline.from_pretrained(
            "black-forest-labs/FLUX.1-dev",
            transformer=None,
            vae=None,
            device_map="balanced",
            max_memory={0: "24GB", 1: "24GB", 2: "24GB", 3: "24GB"},
            torch_dtype=torch.bfloat16
        )
    else:
        print("Using cached encoder pipeline.")
    encoder = loaded_models["encoder"]

    prompt = """
    mirror selfie of a young redhead woman
    """

    with torch.no_grad():
        print("Encoding prompts.")
        prompt_embeds, pooled_prompt_embeds, text_ids = encoder.encode_prompt(
            prompt=prompt, prompt_2=None, max_sequence_length=512
        )

    if "transformer" not in loaded_models:
        print("Loading transformer...")
        transformer = FluxTransformer2DModel.from_pretrained(
            "black-forest-labs/FLUX.1-dev", 
            subfolder="transformer",
            device_map="balanced",
            torch_dtype=torch.bfloat16
        )
        
        loaded_models["transformer"] = transformer
    else:
        print("Using cached transformer.")
    transformer = loaded_models["transformer"]

    if "denoiser" not in loaded_models:
        print("Loading denoiser pipeline...")
        denoiser = FluxPipeline.from_pretrained(
            "black-forest-labs/FLUX.1-dev",
            device_map="balanced",
            text_encoder=None,
            text_encoder_2=None,
            tokenizer=None,
            tokenizer_2=None,
            vae=None,
            transformer=transformer,
            torch_dtype=torch.bfloat16,
        )
        print("Loading loras...")
        start = time.time()
        
        denoiser.load_lora_weights(hf_hub_download("kudzueye/Boreal", "boreal-flux-dev-lora-v04_1000_steps.safetensors"))
        denoiser.fuse_lora()
        # denoiser.unload_lora_weights()

        # denoiser.load_lora_weights(hf_hub_download("ByteDance/Hyper-SD", "Hyper-FLUX.1-dev-8steps-lora.safetensors"))
        # denoiser.fuse_lora(lora_scale=0.125)

        loaded_models["denoiser"] = denoiser
        print(f'Loading loras took {round(time.time() - start)} seconds')
    else:
        print("Using cached denoiser pipeline.")
    denoiser = loaded_models["denoiser"]

    start = time.time()
    print("Running denoising.")
    seed = 1000
    height, width = 1280, 720
    latents = denoiser(
        prompt_embeds=prompt_embeds,
        pooled_prompt_embeds=pooled_prompt_embeds,
        num_inference_steps=30,
        # guidance_scale=0.2,
        guidance_scale=guidance_scale,
        height=height,
        width=width,
        output_type="latent",
        generator=torch.Generator("cuda").manual_seed(seed),
    ).images
    print(f'Denoising took {round(time.time() - start, 3)} seconds')

    if "vae" not in loaded_models:
        print("Loading VAE and image processor...")
        loaded_models["vae"] = AutoencoderKL.from_pretrained(
            "black-forest-labs/FLUX.1-dev", 
            subfolder="vae", 
            torch_dtype=torch.bfloat16,
            device_map="balanced",
        )
        vae_scale_factor = 2 ** (len(loaded_models["vae"].config.block_out_channels))
        loaded_models["image_processor"] = VaeImageProcessor(vae_scale_factor=vae_scale_factor)
    else:
        print("Using cached VAE and image processor.")
    vae = loaded_models["vae"]
    image_processor = loaded_models["image_processor"]

    with torch.no_grad():
        latents = FluxPipeline._unpack_latents(latents, height, width, 8)
        latents = (latents / vae.config.scaling_factor) + vae.config.shift_factor

        image = vae.decode(latents, return_dict=False)[0]
        image = image_processor.postprocess(image, output_type="pil")
        image[0].save(f"{round(guidance_scale, 1)}.jpg")

In [ ]:
i = -8.5
while i < 10:
    generate(guidance_scale=i)
    i += 0.3

In [ ]:
import torch, time
from diffusers import FluxPipeline, FluxTransformer2DModel, AutoencoderKL
from diffusers.image_processor import VaeImageProcessor
from huggingface_hub import hf_hub_download


if "loaded_models" not in globals():
    loaded_models = {}

if "encoder" not in loaded_models:
    print("Loading encoder pipeline...")
    loaded_models["encoder"] = FluxPipeline.from_pretrained(
        "black-forest-labs/FLUX.1-dev", transformer=None, vae=None,
        device_map="balanced", torch_dtype=torch.bfloat16,
        max_memory={0: "24GB", 1: "24GB", 2: "24GB", 3: "24GB"},
    )
else:
    print("Using cached encoder pipeline.")
encoder = loaded_models["encoder"]

prompt = """
photo of kerak emas 
"""

with torch.no_grad():
    print("Encoding prompts.")
    prompt_embeds, pooled_prompt_embeds, text_ids = encoder.encode_prompt(prompt=prompt, prompt_2=None, max_sequence_length=512)

if "transformer" not in loaded_models:
    print("Loading transformer...")
    transformer = FluxTransformer2DModel.from_pretrained(
        "black-forest-labs/FLUX.1-dev", subfolder="transformer",
        device_map="balanced", torch_dtype=torch.bfloat16
    )
    
    loaded_models["transformer"] = transformer
else:
    print("Using cached transformer.")
transformer = loaded_models["transformer"]

if "denoiser" not in loaded_models:
    print("Loading denoiser pipeline...")
    denoiser = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-dev", device_map="balanced", torch_dtype=torch.bfloat16,
        transformer=transformer, text_encoder=None, text_encoder_2=None, tokenizer=None, tokenizer_2=None, vae=None)
    print("Loading loras...")
    start = time.time()
    
    denoiser.load_lora_weights(hf_hub_download("kudzueye/Boreal", "boreal-flux-dev-lora-v04_1000_steps.safetensors"))
    denoiser.fuse_lora()
    # denoiser.unload_lora_weights()

    # denoiser.load_lora_weights(hf_hub_download("ByteDance/Hyper-SD", "Hyper-FLUX.1-dev-8steps-lora.safetensors"))
    # denoiser.fuse_lora(lora_scale=0.125)

    loaded_models["denoiser"] = denoiser
    print(f'Loading loras took {round(time.time() - start)} seconds')
else:
    print("Using cached denoiser pipeline.")
denoiser = loaded_models["denoiser"]

start = time.time()
print("Running denoising.")
seed = 1000
height, width = 768, 768
latents = denoiser(
    prompt_embeds=prompt_embeds, output_type="latent",
    pooled_prompt_embeds=pooled_prompt_embeds,
    num_inference_steps=20,
    guidance_scale=0,
    height=height, width=width,    
    generator=torch.Generator("cuda").manual_seed(seed),
).images
print(f'Denoising took {round(time.time() - start, 2)} seconds')

if "vae" not in loaded_models:
    print("Loading VAE and image processor...")
    loaded_models["vae"] = AutoencoderKL.from_pretrained("black-forest-labs/FLUX.1-dev", subfolder="vae", torch_dtype=torch.bfloat16, device_map="balanced")
    vae_scale_factor = 2 ** (len(loaded_models["vae"].config.block_out_channels))
    loaded_models["image_processor"] = VaeImageProcessor(vae_scale_factor=vae_scale_factor)
else:
    print("Using cached VAE and image processor.")
vae = loaded_models["vae"] 
image_processor = loaded_models["image_processor"]

with torch.no_grad():
    latents = FluxPipeline._unpack_latents(latents, height, width, 8)
    latents = (latents / vae.config.scaling_factor) + vae.config.shift_factor

    image = vae.decode(latents, return_dict=False)[0]
    image = image_processor.postprocess(image, output_type="pil")
    image[0].save("out.jpg")
    image[0].show()

In [ ]:
# !rm -rf ~/.cache/huggingface
# !rm -rf mongo_persistence && mkdir mongo_persistence

In [ ]:
# Downloading models

# !huggingface-cli download mlabonne/Meta-Llama-3.1-8B-Instruct-abliterated-GGUF meta-llama-3.1-8b-instruct-abliterated.Q8_0.gguf --local-dir ~/models/llms
# !huggingface-cli download mlabonne/Meta-Llama-3.1-8B-Instruct-abliterated-GGUF meta-llama-3.1-8b-instruct-abliterated.Q4_K_M.gguf --local-dir ~/models/llms
# !huggingface-cli download mlabonne/Meta-Llama-3.1-8B-Instruct-abliterated-GGUF meta-llama-3.1-8b-instruct-abliterated.Q2_K.gguf --local-dir ~/models/llms

# !huggingface-cli download bartowski/Llama-3.3-70B-Instruct-abliterated-GGUF Llama-3.3-70B-Instruct-abliterated-IQ2_S.gguf --local-dir ~/models/llms

# !huggingface-cli download mradermacher/Qwen2.5-32B-Instruct-abliterated-i1-GGUF Qwen2.5-32B-Instruct-abliterated.i1-Q4_K_S.gguf --local-dir ~/models/llms
# !huggingface-cli download mradermacher/Qwen2.5-32B-Instruct-abliterated-i1-GGUF Qwen2.5-32B-Instruct-abliterated.i1-IQ3_M.gguf --local-dir ~/models/llms

# !huggingface-cli download mradermacher/DeepSeek-R1-Distill-Qwen-32B-abliterated-i1-GGUF DeepSeek-R1-Distill-Qwen-32B-abliterated.i1-Q4_K_M.gguf --local-dir ~/models/llms

In [ ]:
# Setting up LLama.cpp

# !sudo apt install libcurl4-openssl-dev
# !apt-cache policy nvidia-cuda-toolkit
# !sudo apt remove nvidia-cuda-toolkit && sudo apt autoremove

# !git clone https://github.com/ggerganov/llama.cpp --depth 1 --single-branch

# %cd /teamspace/studios/this_studio/llama.cpp
# !cmake -B build -DGGML_CUDA=ON -DGGML_CUDA_F16=ON -DGGML_CUDA_FA_ALL_QUANTS=ON
# !cmake --build build --config Release -j 32
# %cd ~

In [ ]:
# testing llama.cpp tools

# !.~/llama.cpp/build/bin/llama-cli
%cd ~/llama.cpp/build/bin

# !./llama-cli -m ~/llama.gguf -h
# !./llama-cli -m ~/models/llama_6.gguf --gpu-layers 18 --ctx-size 131072 --batch-size 65536 -p "I just " --predict 1024 -no-cnv

!./llama-server -m ~/models/llama_4.gguf --port 8000 --gpu-layers 33 --ctx-size 65536 --batch-size 65536

# !./llama-bench -m ~/models/llama_4.gguf

# !./llama-cli -m ~/models/llama_4.gguf --gpu-layers 33 --ctx-size 94208 --batch-size 65536 -p "Sometimes " --predict 128 -no-cnv

In [ ]:
# short test generation
print(requests.post("http://127.0.0.1:8000/v1/chat/completions", json={"messages": [
    {
      "role": "system",
      "content": "You are a professional sexual romance writer.",
    },
    {
      "role": "user",
      "content": "make up a very explicit sexual scene",          
    }
]}).json()['choices'][0]['message']['content'])

In [ ]:
""" server start command
~/llama.cpp/build/bin/llama-server -m ~/models/llms/llama_8.gguf --port 8000 --gpu-layers 33 --ctx-size 131072 \
--cache-type-k q8_0 --cache-type-v q8_0 --flash-attn
"""

""" start with grammar
~/llama.cpp/build/bin/llama-server -m ~/models/llms/llama_4.gguf --port 8000 --gpu-layers 33 --ctx-size 131072 \
--cache-type-k q4_0 --cache-type-v q4_0 --flash-attn --grammar-file ~/generation/1_bots_text.gbnf
~/llama.cpp/build/bin/llama-server -m ~/models/llms/llama_4.gguf --port 8000 --gpu-layers 33 --ctx-size 131072 \
--cache-type-k q4_0 --cache-type-v q4_0 --flash-attn --grammar-file ~/generation/2_add_img_prompts_to_bots.gbnf
"""

In [ ]:
# installing docker

# !sudo apt install postgresql postgresql-contrib # installing postgres
# !sudo systemctl enable postgresql
# !sudo systemctl start postgresql

# !sudo apt install -y apt-transport-https ca-certificates curl software-properties-common
# !yes | curl -fsSL https://download.docker.com/linux/ubuntu/gpg | sudo gpg --dearmor -o /usr/share/keyrings/docker-archive-keyring.gpg
# !sudo add-apt-repository \"deb [arch=amd64] https://download.docker.com/linux/ubuntu focal stable\"
# !apt-cache policy docker-ce
# !sudo apt install docker-ce
# !sudo systemctl status docker

In [ ]:
# start mongo
!wget -qO- https://www.mongodb.org/static/pgp/server-8.0.asc | sudo tee /etc/apt/trusted.gpg.d/server-8.0.asc
!echo "deb [ arch=amd64,arm64 ] https://repo.mongodb.org/apt/ubuntu focal/mongodb-org/8.0 multiverse" | sudo tee /etc/apt/sources.list.d/mongodb-org-8.0.list
!sudo apt-get update
!sudo apt-get install -y mongodb-mongosh

!docker run --name mongodb -p 27017:27017 -v ~/mongo_persistence:/data/db -d mongo
# mongosh --port 27017
# docker stop mongodb && docker rm mongodb

In [ ]:
## grammar builder https://grammar.intrinsiclabs.ai
# !python3 generation/1_bots_text.py

In [ ]:
# stable-diffusion.cpp

# %cd ~
# !rm -rf stable-diffusion.cpp
# !git clone --recursive https://github.com/leejet/stable-diffusion.cpp
# %cd ~/stable-diffusion.cpp
# !git pull origin master
# !git submodule init
# !git submodule update
# %cd ~

# !huggingface-cli download city96/FLUX.1-dev-gguf flux1-dev-Q8_0.gguf --local-dir ./models/img

# !huggingface-cli download black-forest-labs/FLUX.1-dev ae.safetensors --local-dir ./models/img
# !huggingface-cli download comfyanonymous/flux_text_encoders clip_l.safetensors --local-dir ./models/img
# !huggingface-cli download comfyanonymous/flux_text_encoders t5xxl_fp16.safetensors --local-dir ./models/img
!huggingface-cli download kudzueye/boreal-flux-dev-v2 boreal-v2.safetensors --local-dir ./models/img/loras
!huggingface-cli download kudzueye/Boreal boreal-flux-dev-lora-v04_1000_steps.safetensors --local-dir ./models/img/loras
!mv ./models/img/loras/boreal-flux-dev-lora-v04_1000_steps.safetensors ./models/img/loras/boreal.safetensors
!huggingface-cli download Shakker-Labs/FLUX.1-dev-LoRA-AntiBlur FLUX-dev-lora-AntiBlur.safetensors --local-dir ./models/img/loras
!mv ./models/img/loras/FLUX-dev-lora-AntiBlur.safetensors ./models/img/loras/antiblur.safetensors

# !wget https://github.com/Kitware/CMake/releases/download/v3.31.4/cmake-3.31.4-linux-x86_64.sh
# !chmod +x cmake-3.31.4-linux-x86_64.sh
# !sudo ./cmake-3.31.4-linux-x86_64.sh --prefix=/usr/local --skip-license 
# !rm cmake-3.31.4-linux-x86_64.sh
# !cmake --version

# !mkdir ~/stable-diffusion.cpp/build
# %cd ~/stable-diffusion.cpp/build
# !cmake .. -DSD_CUDA=ON
# !cmake --build . --config Release

# quantizing
# !~/stable-diffusion.cpp/build/bin/sd --mode "convert" --type "q8_0" --model ~/models/img/temp/nr.safetensors

import time
start = time.time()

# prompt = """
# mirror selfie photo of a smiling young redhead woman [<lora:boreal:0.3>|<lora:boreal-v2:0.15>|<lora:antilbur:2>]
# """
prompt = """
selfie photo of a smiling young redhead woman<lora:boreal:1>
"""

# %cd ~/models/img
# !~/stable-diffusion.cpp/build/bin/sd --diffusion-model flux1-dev-Q8_0.gguf --vae ae.safetensors --clip_l clip_l.safetensors \
# --t5xxl t5xxl_fp16.safetensors --steps 20 --cfg-scale 1.0 --sampling-method euler -v --diffusion-fa -W 1024 -H 1280 --seed 42 -p "{prompt}" --type q8_0 \
# --lora-model-dir ./loras --output ~/output.jpg --threads 16

# print(f'took {time.time() - start} seconds')

# %cd ~
# from PIL import Image

# with Image.open("output.png") as img:
#     img.convert("RGB").save("output.jpg", "JPEG")
#     !rm output.png
    
# with Image.open("output.png") as img:
#     img.show()

prompt = """
Night photo of a tall Mediterranean woman with long curly dark hair and a curvy body riding a bike at Times Square
    
[<lora:boreal:0.7>|<lora:boreal-v2:0.15>|<lora:antilbur:2>]
"""


# %cd ~/models/img
# !~/stable-diffusion.cpp/build/bin/sd --mode img2img --diffusion-model flux1-dev-Q8_0.gguf --vae ae.safetensors --clip_l clip_l.safetensors \
# --t5xxl t5xxl_fp16.safetensors --steps 20 --cfg-scale 1.0 --sampling-method euler -v --diffusion-fa -W 1024 -H 1280 --seed 42 -p "{prompt}" --type q8_0 \
# --lora-model-dir ./loras \
# --output ~/output_img.jpg --threads 16 --init-img ~/output.jpg --strength 0.9

In [ ]:
from stable_diffusion_cpp import StableDiffusion
import time

base_path = "./models/img"
start = time.time()
flux = StableDiffusion(
    diffusion_model_path=f"{base_path}/flux1-dev-Q8_0.gguf",
    clip_l_path=f"{base_path}/clip_l.safetensors",
    t5xxl_path=f"{base_path}/t5xxl_fp16.safetensors",
    vae_path=f"{base_path}/ae.safetensors",
    # vae_decode_only=True, # True for txt2img
    diffusion_flash_attn=True,
    lora_model_dir=f"{base_path}/loras/",
)
print(f'\nloading took {time.time() - start}')

In [ ]:
# txt2img
%cd ~
import time
from PIL import Image

start = time.time()
output = flux.txt_to_img(
    prompt="""
    
    Photo of a young redhead woman with white clear skin,
    beautiful Irish features,
    field of tulips in the background, dusk, fireflies in the air
    
    [<lora:boreal:0.5>|<lora:antilbur:5>]""",
    sample_steps=15,
    width=768,
    height=1024,
    seed=1000,
    cfg_scale=0.7,
    guidance=5,
    sample_method="euler",
)
delta = round(time.time() - start)
output[0].convert('RGB').save("output.jpg", "JPEG")
from IPython.display import clear_output
clear_output()
print(f'took {delta} seconds')
with Image.open("output.jpg") as img:
    img.show()

In [ ]:
# img2img
%cd ~
import time
from PIL import Image
width, height = Image.open("output.jpg").size
start = time.time()
# Creepshot of a tall Mediterranean woman with long curly dark hair and a curvy body dining out in a fancy restaurant wearing small black dress.
output = flux.img_to_img(
    prompt="""
    
    Creepshot of a tall Mediterranean woman with long curly dark hair and a curvy body dining out in a fancy restaurant wearing small black dress.
    
    [<lora:boreal:0.7>|<lora:boreal-v2:0.15>|<lora:antilbur:2>]""",
    negative_prompt="worst quality, smooth",
    image="output.jpg", # The input image will be automatically resized to the match the width and height arguments (default: 512x512)
    strength=1,
    sample_steps=20,
    width=width,
    height=height,
    seed=1000,
    cfg_scale=1,
    sample_method="euler",
)
delta = round(time.time() - start)
output[0].convert('RGB').save("output_img.jpg")
from IPython.display import clear_output
clear_output()
print(f'took {delta} seconds')
output[0].show()